# Which weather forecast could I have used?

In the [data engineering notebook](data_engineering1_organized_v3.ipynb) I joined pickups with actual historical weather. That was good for describing what happened. But say I'm making a pickup forecast at noon. Weather at 3 PM is still in the future, so I need a weather *forecast* I could have had at noon.

I start from the saved weather-run selections and follow them into the final forecasting table. The prepared snapshot has all tables I need for these checks. The raw JSON check only runs if I restored the original weather archive.

[Open-Meteo historical forecast and individual runs](https://open-meteo.com/en/docs/historical-forecast-api)


## Why do I need two clocks?

A weather run's start time is not when I could actually use it. I assume a seven-hour delay and pick the newest run that is old enough. Pickup horizon counts from when I make my forecast, but weather lead counts from when the weather run started.

Example: a run that started at 5 AM becomes usable at noon. If at noon I forecast pickups for 3 PM, the pickup horizon is three hours but the weather lead is ten hours. The seven hours is my assumption, and these tables let me check that I really followed it.

These two clocks are easy to mix up. Most chosen runs are 7–12 hours old, but for 54 origins a run was missing and I had to use an older one. I keep those cases, I don't fill them with weather observed later. Checking when runs were actually published in the past is a separate question.

$$
\text{available time}
=
\text{model run time}
+
7\text{ hours}
$$

$$
\text{target hour}
=
\text{forecast origin}
+
\text{forecast horizon}
$$

$$
\text{weather lead}
=
\text{target hour}
-
\text{selected model run time}
$$


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def find_project_root(start: Path) -> Path:
    # Find the project root from the repository or notebooks directory.
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find the project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd())

WEATHER_ROOT = (
    PROJECT_ROOT
    / "data/silver/weather_forecasts/open_meteo_single_runs/ecmwf_ifs"
)
MANIFEST_DIR = WEATHER_ROOT / "manifests"

RUN_MANIFEST_PATH = MANIFEST_DIR / (
    "ecmwf_ifs_required_runs_20240810_20260630_"
    "availability_7h_fh19.parquet"
)
FALLBACK_MANIFEST_PATH = MANIFEST_DIR / (
    "ecmwf_ifs_fallback_origin_manifest_availability_7h.parquet"
)
ORIGIN_SELECTION_PATH = MANIFEST_DIR / (
    "ecmwf_ifs_origin_run_selection_availability_7h_horizon_1_6.parquet"
)
HORIZON_SELECTION_PATH = MANIFEST_DIR / (
    "ecmwf_ifs_origin_horizon_weather_selection_availability_7h.parquet"
)
CLASS_MANIFEST_PATH = MANIFEST_DIR / (
    "nyc_weather_request_class_manifest.parquet"
)

CLASS_PARTITION_ROOT = (
    WEATHER_ROOT
    / "selected_origin_horizon_request_classes/availability_delay=7h"
)
ZONE_PARTITION_ROOT = (
    WEATHER_ROOT
    / "selected_origin_horizon_zones/availability_delay=7h"
)
FINAL_ROOT = (
    PROJECT_ROOT
    / "data/gold/production_model/hourly_zone_horizon_features/"
      "availability_delay=7h"
)
FEATURE_MANIFEST_PATH = (
    FINAL_ROOT / "manifests/production_model_feature_manifest.parquet"
)
SPLIT_MANIFEST_PATH = (
    FINAL_ROOT / "manifests/production_forecast_origin_split_manifest.parquet"
)
FINAL_AUDIT_PATH = (
    FINAL_ROOT / "audit/production_feature_partition_audit.parquet"
)

required_paths = [
    RUN_MANIFEST_PATH,
    FALLBACK_MANIFEST_PATH,
    ORIGIN_SELECTION_PATH,
    HORIZON_SELECTION_PATH,
    CLASS_MANIFEST_PATH,
    FEATURE_MANIFEST_PATH,
    SPLIT_MANIFEST_PATH,
    FINAL_AUDIT_PATH,
]
missing_paths = [path for path in required_paths if not path.is_file()]

print("Project root:", PROJECT_ROOT)
print("Required files found:", len(required_paths) - len(missing_paths), "/", len(required_paths))
print("Missing files:", missing_paths)

assert not missing_paths


Required files found: 8 / 8
Missing files: []


## Which run did I use when a file was missing?

I want three answers: which runs should exist, which were missing, and which run each forecast origin finally used. The manifests answer all three together, so I check they agree before trusting any row counts later.


In [ ]:
run_manifest = pd.read_parquet(RUN_MANIFEST_PATH)
fallback_manifest = pd.read_parquet(FALLBACK_MANIFEST_PATH)
origin_selection = pd.read_parquet(ORIGIN_SELECTION_PATH)

for column in ["model_run_utc"]:
    run_manifest[column] = pd.to_datetime(run_manifest[column], utc=True)

for column in [
    "forecast_origin_utc",
    "fallback_source_model_run_utc",
    "fallback_source_available_utc",
]:
    fallback_manifest[column] = pd.to_datetime(fallback_manifest[column], utc=True)

for column in [
    "forecast_origin_utc",
    "selected_model_run_utc",
    "selected_run_available_utc",
]:
    origin_selection[column] = pd.to_datetime(origin_selection[column], utc=True)

selected_relative_paths = origin_selection["selected_raw_json_relative_path"].drop_duplicates()
selected_files_exist = selected_relative_paths.map(
    lambda value: (PROJECT_ROOT / Path(value)).is_file()
)

status_counts = run_manifest["run_status"].value_counts().to_dict()

print("Scheduled runs:", len(run_manifest))
print("Run status:", status_counts)
print("Fallback origins:", len(fallback_manifest))
print("Fallback source runs:", fallback_manifest["fallback_source_model_run_utc"].nunique())
print("Forecast origins:", len(origin_selection))
print("Selected JSON files:", len(selected_relative_paths))
print("All selected files exist:", selected_files_exist.all())

assert len(run_manifest) == 2_759
assert status_counts == {"complete": 2_750, "unavailable": 9}
assert len(fallback_manifest) == 54
assert fallback_manifest["forecast_origin_utc"].is_unique
assert fallback_manifest["fallback_source_model_run_utc"].nunique() == 6
assert len(origin_selection) == 16_551
assert origin_selection["forecast_origin_utc"].is_unique
assert int(origin_selection["is_fallback_weather_run"].sum()) == 54
assert origin_selection["selected_run_available_utc"].le(
    origin_selection["forecast_origin_utc"]
).all()
CHECK_RAW_ARCHIVE = False  # Set True after restoring the original weather JSON.
if CHECK_RAW_ARCHIVE:
    assert selected_files_exist.all()
else:
    print("Raw JSON availability is informational in the prepared snapshot.")

print("Archive and origin-selection validation passed: True")


Scheduled runs: 2759
Run status: {'complete': 2750, 'unavailable': 9}
Fallback origins: 54
Fallback source runs: 6
Forecast origins: 16551
Selected JSON files: 2756
All selected files exist: False
Raw JSON availability is informational in the prepared snapshot.
Archive and origin-selection validation passed: True


## Does each forecast cover the next six hours?

Each origin predicts the next six hours. With 16,551 origins that gives 99,306 origin-horizon pairs before adding zones. I check keys and timestamp math here, because a wrong horizon would pass through every later join without anyone noticing.

$$
16{,}551 \times 6 = 99{,}306
$$


In [ ]:
horizon_selection = pd.read_parquet(HORIZON_SELECTION_PATH)

for column in [
    "forecast_origin_utc",
    "target_hour_utc",
    "selected_model_run_utc",
    "selected_run_available_utc",
]:
    horizon_selection[column] = pd.to_datetime(horizon_selection[column], utc=True)

duplicate_keys = horizon_selection.duplicated(
    ["forecast_origin_utc", "forecast_horizon"]
).sum()
rows_per_horizon = horizon_selection["forecast_horizon"].value_counts().sort_index()

target_arithmetic_valid = horizon_selection["target_hour_utc"].eq(
    horizon_selection["forecast_origin_utc"]
    + pd.to_timedelta(horizon_selection["forecast_horizon"], unit="h")
)
lead_arithmetic_valid = horizon_selection["weather_lead_hours"].eq(
    horizon_selection["selected_run_age_hours"]
    + horizon_selection["forecast_horizon"]
)

print("Rows:", len(horizon_selection))
print("Unique origins:", horizon_selection["forecast_origin_utc"].nunique())
print("Duplicate keys:", duplicate_keys)
print("Rows per horizon:", rows_per_horizon.to_dict())
print("Fallback rows:", int(horizon_selection["is_fallback_weather_run"].sum()))
print("Weather-lead range:", (
    int(horizon_selection["weather_lead_hours"].min()),
    int(horizon_selection["weather_lead_hours"].max()),
))

assert len(horizon_selection) == 99_306
assert horizon_selection["forecast_origin_utc"].nunique() == 16_551
assert duplicate_keys == 0
assert rows_per_horizon.eq(16_551).all()
assert target_arithmetic_valid.all()
assert lead_arithmetic_valid.all()
assert horizon_selection["selected_run_available_utc"].le(
    horizon_selection["forecast_origin_utc"]
).all()
assert horizon_selection["weather_lead_hours"].lt(
    horizon_selection["selected_forecast_hours"]
).all()

print("Origin–horizon validation passed: True")


Rows: 99306
Unique origins: 16551
Duplicate keys: 0
Rows per horizon: {1: 16551, 2: 16551, 3: 16551, 4: 16551, 5: 16551, 6: 16551}
Fallback rows: 324
Weather-lead range: (8, 36)
Origin–horizon validation passed: True


## How do 28 weather requests cover 263 zones?

Asking for the same weather separately for neighboring zones is extra work with no new information. So I use 28 request classes and then expand them to all 263 zones. Zones 57, 104 and 105 use nearby zones' coordinates as stand-ins, and I check that these replacements survive the expansion and every zone gets its rows.

$$
99{,}306 \times 28 = 2{,}780{,}568
$$

$$
99{,}306 \times 263 = 26{,}117{,}478
$$


In [ ]:
class_manifest = pd.read_parquet(CLASS_MANIFEST_PATH)

class_partition_paths = sorted(
    CLASS_PARTITION_ROOT.glob("year=*/month=*/selected_weather_request_classes_*.parquet")
)
zone_partition_paths = sorted(
    ZONE_PARTITION_ROOT.glob("year=*/month=*/selected_weather_zones_*.parquet")
)

class_rows = sum(pq.ParquetFile(path).metadata.num_rows for path in class_partition_paths)
zone_rows = sum(pq.ParquetFile(path).metadata.num_rows for path in zone_partition_paths)

proxy_mapping = dict(
    class_manifest.loc[
        class_manifest["is_coordinate_proxy"],
        ["location_id", "coordinate_source_location_id"],
    ].itertuples(index=False, name=None)
)

print("Manifest rows:", len(class_manifest))
print("Weather classes:", class_manifest["weather_request_class"].nunique())
print("Representative zones:", int(class_manifest["is_representative"].sum()))
print("Coordinate proxies:", proxy_mapping)
print("Class partitions:", len(class_partition_paths), "| rows:", class_rows)
print("Zone partitions:", len(zone_partition_paths), "| rows:", zone_rows)

assert len(class_manifest) == 263
assert class_manifest["location_id"].nunique() == 263
assert set(class_manifest["location_id"]) == set(range(1, 264))
assert class_manifest["weather_request_class"].nunique() == 28
assert class_manifest.groupby("weather_request_class")["is_representative"].sum().eq(1).all()
assert proxy_mapping == {57: 56, 104: 103, 105: 103}
assert len(class_partition_paths) == 23
assert len(zone_partition_paths) == 23
assert class_rows == 2_780_568
assert zone_rows == 26_117_478

print("Weather-class and zone-expansion validation passed: True")


Manifest rows: 263
Weather classes: 28
Representative zones: 28
Coordinate proxies: {104: 103, 105: 103, 57: 56}
Class partitions: 23 | rows: 2780568
Zone partitions: 23 | rows: 26117478
Weather-class and zone-expansion validation passed: True


## Did the joins keep one row for each forecast?

Now I check the table that brings together weather, pickup targets, lags, calendar and events. A correct merge keeps exactly one row per origin, horizon and zone.

Two details stay visible. The repeated 1 AM hour in November is flagged, because it maps differently from normal hours. And incomplete forecast values stay missing. The flag shows where the daylight-saving issue is, but it doesn't undo the neighboring-week fix I made earlier.


In [ ]:
final_partition_paths = sorted(
    FINAL_ROOT.glob("year=*/month=*/production_features_*.parquet")
)
final_audit = pd.read_parquet(FINAL_AUDIT_PATH)

schema_records = []
for path in final_partition_paths:
    parquet_file = pq.ParquetFile(path)
    schema = parquet_file.schema_arrow
    schema_records.append({
        "rows": parquet_file.metadata.num_rows,
        "names": tuple(schema.names),
        "types": tuple(str(field.type) for field in schema),
        "size_mb": path.stat().st_size / 1024**2,
    })

schema_audit = pd.DataFrame(schema_records)
reference_names = schema_audit.loc[0, "names"]
reference_types = schema_audit.loc[0, "types"]
same_names = schema_audit["names"].map(lambda value: value == reference_names).all()
same_types = schema_audit["types"].map(lambda value: value == reference_types).all()

total_rows = int(schema_audit["rows"].sum())
duplicate_keys = int(final_audit["duplicate_keys"].sum())
missing_context_rows = int(final_audit["missing_context_rows"].sum())
ambiguous_dst_rows = int(final_audit["ambiguous_dst_rows"].sum())

print("Partitions:", len(final_partition_paths))
print("Rows:", total_rows)
print("Columns:", len(reference_names))
print("Identical schemas:", same_names and same_types)
print("Duplicate keys:", duplicate_keys)
print("Missing calendar/event rows:", missing_context_rows)
print("Ambiguous DST rows:", ambiguous_dst_rows)
print("Post-game rows:", int(final_audit["post_game_rows"].sum()))
print("Concert-window rows:", int(final_audit["concert_window_rows"].sum()))
print("Wide-concert-window rows:", int(final_audit["wide_concert_window_rows"].sum()))
print("Total size (MB):", round(schema_audit["size_mb"].sum(), 2))

assert len(final_partition_paths) == 23
assert total_rows == 26_117_478
assert len(reference_names) == 53
assert same_names and same_types
assert duplicate_keys == 0
assert missing_context_rows == 0
assert ambiguous_dst_rows == 6_312

print("Final production dataset validation passed: True")


Partitions: 23
Rows: 26117478
Columns: 53
Identical schemas: True
Duplicate keys: 0
Missing calendar/event rows: 0
Ambiguous DST rows: 6312
Post-game rows: 6437
Concert-window rows: 10248
Wide-concert-window rows: 15372
Total size (MB): 126.9
Final production dataset validation passed: True


In [ ]:
weather_columns = [
    "forecast_temperature_f",
    "forecast_precipitation_in",
    "forecast_rain_in",
    "forecast_snowfall_in",
    "forecast_wind_speed_mph",
]

missing_weather = pd.Series(0, index=weather_columns, dtype="int64")
incomplete_weather_rows = 0

for path in final_partition_paths:
    weather = pd.read_parquet(path, columns=weather_columns)
    missing_weather = missing_weather.add(weather.isna().sum(), fill_value=0).astype("int64")
    incomplete_weather_rows += int(weather.isna().any(axis=1).sum())

print("Incomplete weather rows:", incomplete_weather_rows)
print("Percentage of final rows:", round(100 * incomplete_weather_rows / total_rows, 4))
print("\nMissing values by variable:")
print(missing_weather)

assert incomplete_weather_rows == 217_764
assert missing_weather.to_dict() == {
    "forecast_temperature_f": 94_680,
    "forecast_precipitation_in": 142_020,
    "forecast_rain_in": 189_360,
    "forecast_snowfall_in": 94_680,
    "forecast_wind_speed_mph": 160_956,
}

print("Weather-missingness validation passed: True")


Incomplete weather rows: 217764
Percentage of final rows: 0.8338

Missing values by variable:
forecast_temperature_f        94680
forecast_precipitation_in    142020
forecast_rain_in             189360
forecast_snowfall_in          94680
forecast_wind_speed_mph      160956
dtype: int64
Weather-missingness validation passed: True


## Which columns should the model actually see?

The final table has 53 columns, and putting all of them into a model would be a mistake. Some only identify the source or help explain data quality. I use the manifest's list of 30 features and their categorical roles, so the model always gets the intended inputs in the same order.


In [ ]:
feature_manifest = pd.read_parquet(FEATURE_MANIFEST_PATH)

role_counts = feature_manifest["role"].value_counts().to_dict()
primary_features = feature_manifest.loc[
    feature_manifest["use_in_primary_model"], "column_name"
].tolist()
categorical_features = feature_manifest.loc[
    feature_manifest["use_in_primary_model"]
    & feature_manifest["modeling_type"].eq("categorical"),
    "column_name",
].tolist()

unexpected_observed_weather = sorted(
    set(["temperature_f", "precipitation_in", "rain_in", "snowfall_in", "wind_speed_mph"])
    & set(reference_names)
)

print("Manifest rows:", len(feature_manifest))
print("Role counts:", role_counts)
print("Primary features:", len(primary_features))
print("Categorical features:", categorical_features)
print("Unexpected observed-weather columns:", unexpected_observed_weather)

assert len(feature_manifest) == 53
assert feature_manifest["column_name"].is_unique
assert set(feature_manifest["column_name"]) == set(reference_names)
assert role_counts == {
    "primary_feature": 30,
    "excluded_metadata": 18,
    "key": 3,
    "sensitivity_flag": 1,
    "target": 1,
}
assert len(primary_features) == 30
assert not unexpected_observed_weather

print("Feature-manifest validation passed: True")


Manifest rows: 53
Role counts: {'primary_feature': 30, 'excluded_metadata': 18, 'key': 3, 'sensitivity_flag': 1, 'target': 1}
Primary features: 30
Categorical features: ['day_of_week', 'forecast_horizon', 'hour_of_day', 'location_id', 'month']
Unexpected observed-weather columns: []
Feature-manifest validation passed: True


## Why leave a gap between time splits?

Three date ranges are not enough. Each origin predicts six hours ahead, so the last origins before a boundary would share targets with the next period. So I remove six origins at each boundary.

Training ends in December 2025, validation is January–March 2026, and test is April–June. This separates the final model-development windows, although my earlier event exploration already used dates in the test period.


In [ ]:
split_manifest = pd.read_parquet(SPLIT_MANIFEST_PATH)

for column in [
    "forecast_origin_utc",
    "target_start_utc",
    "target_end_utc",
]:
    split_manifest[column] = pd.to_datetime(split_manifest[column], utc=True)

split_counts = split_manifest["split"].value_counts().to_dict()

training = split_manifest[split_manifest["split"].eq("training")]
validation = split_manifest[split_manifest["split"].eq("validation")]
test = split_manifest[split_manifest["split"].eq("test")]

training_validation_separated = (
    training["target_end_utc"].max() < validation["target_start_utc"].min()
)
validation_test_separated = (
    validation["target_end_utc"].max() < test["target_start_utc"].min()
)

print("Origin counts:", split_counts)
print("Training and validation separated:", training_validation_separated)
print("Validation and test separated:", validation_test_separated)
print("Total origins:", len(split_manifest))
print("Represented rows:", int(split_manifest["rows"].sum()))

assert len(split_manifest) == 16_551
assert split_manifest["forecast_origin_utc"].is_unique
assert split_counts == {
    "training": 12_208,
    "test": 2_178,
    "validation": 2_153,
    "purged_boundary": 12,
}
assert training_validation_separated
assert validation_test_separated
assert int(split_manifest["rows"].sum()) == 26_117_478

print("Purged split validation passed: True")


Origin counts: {'training': 12208, 'test': 2178, 'validation': 2153, 'purged_boundary': 12}
Training and validation separated: True
Validation and test separated: True
Total origins: 16551
Represented rows: 26117478
Purged split validation passed: True


## What can I take into modeling?

The checks cover 23 monthly files and about 26.1 million rows before purging. Keys are unique, and targets, lags, calendar and event fields are complete. Weather is missing in 217,764 rows, so the model needs to handle that.

Next, the [forecasting data exploration](03b_production_dataset_eda.ipynb) shows where these missing weather values are. The [modeling notebook](04_production_modeling_refactored_v2.ipynb) uses the same feature list and time splits. For a live version I would still need timely pickup counts and a check of when weather runs were really published.
